In [1]:
# Spotlight - Baseline Recommender
# Notebook 02

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

print("Baseline recommender notebook ready!")

Baseline recommender notebook ready!


In [2]:
# Load KuaiRec interaction data

small_matrix = pd.read_csv("../data/small_matrix.csv")

# Create engagement score
small_matrix["watch_score"] = (
    small_matrix["watch_ratio"].clip(upper=3.0)
)

# Remove interactions with missing dates
small_matrix = small_matrix.dropna(subset=["date"]).copy()

# Convert date column to datetime
small_matrix["date"] = pd.to_datetime(small_matrix["date"])

# Temporal train/test split
train_end = pd.Timestamp("2020-08-29")

train_data = small_matrix[
    small_matrix["date"] <= train_end
].copy()

test_data = small_matrix[
    small_matrix["date"] > train_end
].copy()

print("Training interactions:", len(train_data))
print("Testing interactions:", len(test_data))
print("Training users:", train_data["user_id"].nunique())
print("Training videos:", train_data["video_id"].nunique())

Training interactions: 4494578
Testing interactions: 0
Training users: 1411
Training videos: 3327


In [3]:
print("Date minimum:", small_matrix["date"].min())
print("Date maximum:", small_matrix["date"].max())

print("\nLast 10 dates:")
print(small_matrix["date"].sort_values().tail(10).to_list())

print("\nTrain end:", train_end)


Date minimum: 1970-01-01 00:00:00.020200705
Date maximum: 1970-01-01 00:00:00.020200905

Last 10 dates:
[Timestamp('1970-01-01 00:00:00.020200905'), Timestamp('1970-01-01 00:00:00.020200905'), Timestamp('1970-01-01 00:00:00.020200905'), Timestamp('1970-01-01 00:00:00.020200905'), Timestamp('1970-01-01 00:00:00.020200905'), Timestamp('1970-01-01 00:00:00.020200905'), Timestamp('1970-01-01 00:00:00.020200905'), Timestamp('1970-01-01 00:00:00.020200905'), Timestamp('1970-01-01 00:00:00.020200905'), Timestamp('1970-01-01 00:00:00.020200905')]

Train end: 2020-08-29 00:00:00


In [5]:
# Reload the original data to undo the incorrect date conversion

small_matrix = pd.read_csv("../data/small_matrix.csv")

# Create engagement score
small_matrix["watch_score"] = (
    small_matrix["watch_ratio"].clip(upper=3.0)
)

# Remove missing dates
small_matrix = small_matrix.dropna(subset=["date"]).copy()

# Convert the original YYYYMMDD values correctly
small_matrix["date"] = pd.to_datetime(
    small_matrix["date"].astype(int).astype(str),
    format="%Y%m%d"
)

# Temporal train/test split
train_end = pd.Timestamp("2020-08-29")

train_data = small_matrix[
    small_matrix["date"] <= train_end
].copy()

test_data = small_matrix[
    small_matrix["date"] > train_end
].copy()

print("Date minimum:", small_matrix["date"].min())
print("Date maximum:", small_matrix["date"].max())

print("\nTraining interactions:", len(train_data))
print("Testing interactions:", len(test_data))
print("Training users:", train_data["user_id"].nunique())
print("Training videos:", train_data["video_id"].nunique())

Date minimum: 2020-07-05 00:00:00
Date maximum: 2020-09-05 00:00:00

Training interactions: 4387382
Testing interactions: 107196
Training users: 1411
Training videos: 3288


In [6]:
# Build the user-item interaction matrix

user_item_matrix = train_data.pivot_table(
    index="user_id",
    columns="video_id",
    values="watch_score",
    aggfunc="mean",
    fill_value=0
)

print("User-item matrix shape:", user_item_matrix.shape)
print("Number of users:", user_item_matrix.shape[0])
print("Number of videos:", user_item_matrix.shape[1])

User-item matrix shape: (1411, 3288)
Number of users: 1411
Number of videos: 3288


In [7]:
# Calculate user-user similarity

user_similarity = cosine_similarity(user_item_matrix)

print("User similarity matrix shape:", user_similarity.shape)

User similarity matrix shape: (1411, 1411)


In [8]:
# Find the most similar users to User 14

target_user_id = 14

target_index = user_item_matrix.index.get_loc(target_user_id)

similarities = user_similarity[target_index]

# Exclude User 14 itself
similar_indices = np.argsort(similarities)[::-1][1:11]

similar_users = pd.DataFrame({
    "user_id": user_item_matrix.index[similar_indices],
    "similarity": similarities[similar_indices]
})

print("Most similar users to User 14:")
print(similar_users)

Most similar users to User 14:
   user_id  similarity
0     1584    0.857664
1     1641    0.856157
2     4819    0.855973
3     3948    0.855957
4     1704    0.855460
5     3223    0.853768
6      840    0.853494
7     2143    0.853348
8     6927    0.853343
9     2118    0.852978


In [9]:
# Generate baseline recommendations for User 14

top_n_similar = 20

# Get the 20 most similar users
similar_indices = np.argsort(similarities)[::-1][1:top_n_similar + 1]

similar_user_ids = user_item_matrix.index[similar_indices]
similarity_scores = similarities[similar_indices]

# Get their watch scores
similar_user_matrix = user_item_matrix.loc[similar_user_ids]

# Weight each user's watch scores by similarity
weighted_scores = similar_user_matrix.T.dot(similarity_scores)

# Normalize by total similarity
recommendation_scores = weighted_scores / similarity_scores.sum()

# Remove videos User 14 has already watched
watched_videos = set(
    train_data.loc[
        train_data["user_id"] == target_user_id,
        "video_id"
    ]
)

recommendation_scores = recommendation_scores[
    ~recommendation_scores.index.isin(watched_videos)
]

# Get top 10 recommendations
top_recommendations = recommendation_scores.sort_values(
    ascending=False
).head(10)

print("Top 10 baseline recommendations for User 14:")
print(top_recommendations)

Top 10 baseline recommendations for User 14:
video_id
3107     1.464046
152      1.245360
3992     1.193963
2065     1.165342
6177     1.157445
6086     1.116973
10046    1.089688
835      1.082590
6308     1.078616
7263     1.070195
dtype: float64


In [10]:
# Check User 14's actual interactions in the test period

user_test_data = test_data[
    test_data["user_id"] == target_user_id
].copy()

print("User 14 test interactions:", len(user_test_data))

print("\nVideos User 14 actually watched in the test period:")
print(
    user_test_data[
        ["video_id", "watch_ratio", "watch_score", "date"]
    ].head(20)
)

User 14 test interactions: 76

Videos User 14 actually watched in the test period:
      video_id  watch_ratio  watch_score       date
3158      1564     1.346148     1.346148 2020-08-30
3159      3266     0.233166     0.233166 2020-08-30
3160      6307     0.254805     0.254805 2020-08-30
3161      3141     0.560875     0.560875 2020-08-30
3162      7313     0.531873     0.531873 2020-08-30
3163      1572     0.880371     0.880371 2020-08-30
3164      9284     0.497099     0.497099 2020-08-30
3165      6308     1.632857     1.632857 2020-08-30
3166      1554     0.072496     0.072496 2020-08-30
3167      6109     0.233748     0.233748 2020-08-30
3168      1433     1.198574     1.198574 2020-08-30
3169     10377     1.026156     1.026156 2020-08-30
3170      6422     0.442592     0.442592 2020-08-30
3171      4875     0.999644     0.999644 2020-08-30
3172      3117     1.525608     1.525608 2020-08-30
3173      1404     0.816399     0.816399 2020-08-30
3174      1565     1.382536     1

In [11]:
# Evaluate Top-10 recommendations for User 14

recommended_videos = set(top_recommendations.index)
actual_test_videos = set(user_test_data["video_id"])

hits = recommended_videos.intersection(actual_test_videos)

print("Recommended videos:", recommended_videos)
print("Actual test videos:", actual_test_videos)

print("\nHits:", hits)
print("Number of hits:", len(hits))

hit_rate_at_10 = len(hits) / len(recommended_videos)

print("Hit Rate@10:", hit_rate_at_10)

Recommended videos: {6177, 3107, 835, 6308, 6086, 2065, 3992, 152, 10046, 7263}
Actual test videos: {7936, 4987, 1667, 6533, 10377, 3338, 4875, 7693, 7566, 3344, 7313, 1554, 3857, 6422, 6296, 1433, 1564, 1565, 3360, 6307, 1572, 6308, 3109, 7975, 3112, 6441, 5027, 1579, 6316, 3117, 1583, 8624, 10291, 3125, 7734, 3382, 10552, 7354, 1596, 3133, 3265, 3266, 6467, 9284, 3141, 4932, 2373, 2759, 3018, 9291, 1612, 7887, 4944, 9299, 1622, 3288, 3289, 3290, 6745, 6109, 3293, 10589, 10595, 7932, 6244, 6502, 3304, 4712, 3307, 1643, 4978, 10099, 6267, 1404, 1405, 4222}

Hits: {6308}
Number of hits: 1
Hit Rate@10: 0.1


In [12]:
# Evaluate baseline Hit Rate@10 across all users

results = []

top_n_similar = 20
top_k = 10

for user_id in user_item_matrix.index:

    # Find user's position in similarity matrix
    user_index = user_item_matrix.index.get_loc(user_id)

    similarities = user_similarity[user_index]

    # Get top 20 similar users, excluding the user themselves
    similar_indices = np.argsort(similarities)[::-1][1:top_n_similar + 1]

    similar_user_ids = user_item_matrix.index[similar_indices]
    similarity_scores = similarities[similar_indices]

    # Get their watch scores
    similar_user_matrix = user_item_matrix.loc[similar_user_ids]

    # Calculate similarity-weighted scores
    weighted_scores = similar_user_matrix.T.dot(similarity_scores)
    recommendation_scores = weighted_scores / similarity_scores.sum()

    # Remove videos already watched by this user in training
    watched_videos = set(
        train_data.loc[
            train_data["user_id"] == user_id,
            "video_id"
        ]
    )

    recommendation_scores = recommendation_scores[
        ~recommendation_scores.index.isin(watched_videos)
    ]

    # Top 10 recommendations
    recommendations = recommendation_scores.nlargest(top_k).index

    # Actual test videos
    actual_videos = set(
        test_data.loc[
            test_data["user_id"] == user_id,
            "video_id"
        ]
    )

    # Calculate hits
    hits = set(recommendations).intersection(actual_videos)

    hit_rate = len(hits) / top_k

    results.append({
        "user_id": user_id,
        "hits": len(hits),
        "hit_rate_at_10": hit_rate
    })

baseline_results = pd.DataFrame(results)

print("Users evaluated:", len(baseline_results))
print("Average Hit Rate@10:",
      baseline_results["hit_rate_at_10"].mean())
print("Users with at least one hit:",
      (baseline_results["hits"] > 0).sum())

Users evaluated: 1411
Average Hit Rate@10: 0.14776754075124027
Users with at least one hit: 994


In [14]:
# Check the columns in item_daily_features

print("Columns in item_daily_features:")
print(item_features.columns.tolist())

Columns in item_daily_features:
['video_id', 'date', 'author_id', 'video_type', 'upload_dt', 'upload_type', 'visible_status', 'video_duration', 'video_width', 'video_height', 'music_id', 'video_tag_id', 'video_tag_name', 'show_cnt', 'show_user_num', 'play_cnt', 'play_user_num', 'play_duration', 'complete_play_cnt', 'complete_play_user_num', 'valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt', 'long_time_play_user_num', 'short_time_play_cnt', 'short_time_play_user_num', 'play_progress', 'comment_stay_duration', 'like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt', 'cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt', 'comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt', 'delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt', 'comment_like_user_num', 'follow_cnt', 'follow_user_num', 'cancel_follow_cnt', 'cancel_follow_user_num', 'share_cnt', 'share_user_num', 'download_cnt', 'download_user_num', 'report_cnt', 'report_user_num', 'reduce_sim

In [15]:
print("Variables currently available:")
print([x for x in [
    "creator_stats",
    "creator_threshold",
    "video_creator",
    "item_features"
] if x in globals()])


Variables currently available:
['video_creator', 'item_features']


In [16]:
# Check which KuaiRec file contains creator-level information

import os

for filename in os.listdir("../data"):
    if filename.endswith(".csv"):
        df_temp = pd.read_csv("../data/" + filename, nrows=2)
        print(filename, "->", df_temp.columns.tolist())

big_matrix.csv -> ['user_id', 'video_id', 'play_duration', 'video_duration', 'time', 'date', 'timestamp', 'watch_ratio']
item_categories.csv -> ['video_id', 'feat']
item_daily_features.csv -> ['video_id', 'date', 'author_id', 'video_type', 'upload_dt', 'upload_type', 'visible_status', 'video_duration', 'video_width', 'video_height', 'music_id', 'video_tag_id', 'video_tag_name', 'show_cnt', 'show_user_num', 'play_cnt', 'play_user_num', 'play_duration', 'complete_play_cnt', 'complete_play_user_num', 'valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt', 'long_time_play_user_num', 'short_time_play_cnt', 'short_time_play_user_num', 'play_progress', 'comment_stay_duration', 'like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt', 'cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt', 'comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt', 'delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt', 'comment_like_user_num', 'follow_cnt', 'follow_user_num',

In [17]:
# Recreate creator-level total shows

creator_stats = (
    item_features.groupby("author_id")["show_cnt"]
    .sum()
    .reset_index(name="total_shows")
)

print("Number of creators:", len(creator_stats))

print("\nCreator total shows:")
print(creator_stats["total_shows"].describe())

Number of creators: 8369

Creator total shows:
count    8.369000e+03
mean     3.866465e+06
std      1.452844e+07
min      3.000000e+00
25%      4.354000e+03
50%      4.709300e+04
75%      8.410930e+05
max      3.750626e+08
Name: total_shows, dtype: float64


In [18]:
# Restrict creator statistics to creators represented in our experiment

experiment_creators = video_creator["author_id"].dropna().unique()

creator_stats_exp = creator_stats[
    creator_stats["author_id"].isin(experiment_creators)
].copy()

print("Creators represented in experiment:", len(creator_stats_exp))

print("\nCreator total shows:")
print(creator_stats_exp["total_shows"].describe())

print("\n25th percentile:",
      creator_stats_exp["total_shows"].quantile(0.25))

Creators represented in experiment: 8369

Creator total shows:
count    8.369000e+03
mean     3.866465e+06
std      1.452844e+07
min      3.000000e+00
25%      4.354000e+03
50%      4.709300e+04
75%      8.410930e+05
max      3.750626e+08
Name: total_shows, dtype: float64

25th percentile: 4354.0


In [19]:
# Find creators represented by videos in our interaction dataset

experiment_video_ids = small_matrix["video_id"].unique()

experiment_creators = (
    video_creator[
        video_creator["video_id"].isin(experiment_video_ids)
    ]["author_id"]
    .dropna()
    .unique()
)

creator_stats_exp = creator_stats[
    creator_stats["author_id"].isin(experiment_creators)
].copy()

print("Videos in experiment:", len(experiment_video_ids))
print("Creators represented in experiment:", len(creator_stats_exp))

print("\n25th percentile:",
      creator_stats_exp["total_shows"].quantile(0.25))

Videos in experiment: 3327
Creators represented in experiment: 2031

25th percentile: 1139461.5


In [20]:
# Define Emerging and Established creators

creator_threshold = creator_stats_exp["total_shows"].quantile(0.25)

creator_stats_exp["creator_group"] = np.where(
    creator_stats_exp["total_shows"] <= creator_threshold,
    "Emerging",
    "Established"
)

print("Creator threshold:", creator_threshold)

print("\nCreator groups:")
print(
    creator_stats_exp["creator_group"].value_counts()
)

print("\nCreator group percentages:")
print(
    creator_stats_exp["creator_group"]
    .value_counts(normalize=True)
    .mul(100)
)

Creator threshold: 1139461.5

Creator groups:
creator_group
Established    1523
Emerging        508
Name: count, dtype: int64

Creator group percentages:
creator_group
Established    74.987691
Emerging       25.012309
Name: proportion, dtype: float64


In [22]:
# Build Top-10 baseline recommendations for all users

all_recommendations = []

top_n_similar = 20
top_k = 10

for user_id in user_item_matrix.index:

    # Find user's position in similarity matrix
    user_index = user_item_matrix.index.get_loc(user_id)
    similarities = user_similarity[user_index]

    # Top 20 similar users
    similar_indices = np.argsort(similarities)[::-1][1:top_n_similar + 1]

    similar_user_ids = user_item_matrix.index[similar_indices]
    similarity_scores = similarities[similar_indices]

    # Their watch scores
    similar_user_matrix = user_item_matrix.loc[similar_user_ids]

    # Similarity-weighted recommendation scores
    weighted_scores = similar_user_matrix.T.dot(similarity_scores)
    recommendation_scores = (
        weighted_scores / similarity_scores.sum()
    )

    # Remove videos already watched in training
    watched_videos = set(
        train_data.loc[
            train_data["user_id"] == user_id,
            "video_id"
        ]
    )

    recommendation_scores = recommendation_scores[
        ~recommendation_scores.index.isin(watched_videos)
    ]

    # Top 10
    recommendations = recommendation_scores.nlargest(top_k)

    for rank, (video_id, score) in enumerate(
        recommendations.items(), start=1
    ):
        all_recommendations.append({
            "user_id": user_id,
            "video_id": video_id,
            "rank": rank,
            "score": score
        })

baseline_recommendations = pd.DataFrame(all_recommendations)

print("Users:", baseline_recommendations["user_id"].nunique())
print("Recommendation slots:", len(baseline_recommendations))
print("Recommendations per user:",
      baseline_recommendations.groupby("user_id").size().unique())

Users: 1411
Recommendation slots: 14110
Recommendations per user: [10]


In [23]:
    similar_indices = np.argsort(similarities)[::-1][1:top_n_similar + 1]

    similar_user_ids = user_item_matrix.index[similar_indices]
    similarity_scores = similarities[similar_indices]

    # Their watch scores
    similar_user_matrix = user_item_matrix.loc[similar_user_ids]

    # Similarity-weighted recommendation scores
    weighted_scores = similar_user_matrix.T.dot(similarity_scores)
    recommendation_scores = (
        weighted_scores / similarity_scores.sum()
    )

    # Remove videos already watched in training
    watched_videos = set(
        train_data.loc[
            train_data["user_id"] == user_id,
            "video_id"
        ]
    )

    recommendation_scores = recommendation_scores[
        ~recommendation_scores.index.isin(watched_videos)
    ]

    # Top 10
    recommendations = recommendation_scores.nlargest(top_k)

    for rank, (video_id, score) in enumerate(
        recommendations.items(), start=1
    ):
        all_recommendations.append({
            "user_id": user_id,
            "video_id": video_id,
            "rank": rank,
            "score": score
        })

baseline_recommendations = pd.DataFrame(all_recommendations)

print("Users:", baseline_recommendations["user_id"].nunique())
print("Recommendation slots:", len(baseline_recommendations))
print("Recommendations per user:",
      baseline_recommendations.groupby("user_id").size().unique())

Users: 1411
Recommendation slots: 14120
Recommendations per user: [10 20]


In [24]:
# Calculate creator exposure from baseline recommendations

baseline_recommendations = baseline_recommendations.merge(
    video_creator_groups[["video_id", "author_id", "creator_group"]],
    on="video_id",
    how="left"
)

print("Total recommendation slots:",
      len(baseline_recommendations))

print("\nMissing creator groups:",
      baseline_recommendations["creator_group"].isna().sum())

print("\nBaseline creator exposure:")
print(
    baseline_recommendations["creator_group"].value_counts()
)

print("\nBaseline exposure percentage:")
print(
    baseline_recommendations["creator_group"]
    .value_counts(normalize=True).mul(100)
)

Total recommendation slots: 14120

Missing creator groups: 0

Baseline creator exposure:
creator_group
Established    10325
Emerging        3795
Name: count, dtype: int64

Baseline exposure percentage:
creator_group
Established    73.123229
Emerging       26.876771
Name: proportion, dtype: float64
